# Deepfake Face Swap - Google Colab / Kaggle

Notebook ini menjalankan web Flask untuk face swap di video menggunakan InsightFace.

**Langkah:**
1. Clone repository
2. Install dependencies
3. Jalankan web server (dengan ngrok untuk akses dari browser)

## 1. Clone Repository

Ganti `YOUR_GITHUB_USERNAME` dan `YOUR_REPO_NAME` dengan URL repository Anda.

Contoh: `https://github.com/username/deepfake_swapface_in_video`

In [ ]:
# Ganti dengan URL repo Anda
REPO_URL = "https://github.com/YOUR_USERNAME/deepfake_swapface_in_video.git"

!git clone {REPO_URL}
%cd deepfake_swapface_in_video

## 2. Install Dependencies

In [ ]:
# Install base requirements
!pip install -q flask insightface opencv-python-headless numpy Pillow werkzeug huggingface_hub

# Deteksi GPU dan install onnxruntime yang sesuai
# Di Kaggle/Colab: pastikan GPU/Accelerator sudah diaktifkan di Settings
import subprocess
import sys
has_gpu = False
try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True, timeout=5)
        has_gpu = True
    except Exception:
        pass

if has_gpu:
    print("GPU terdeteksi - menginstall onnxruntime-gpu...")
    !pip uninstall -y onnxruntime 2>/dev/null || true
    !pip install -q onnxruntime-gpu
else:
    print("GPU tidak terdeteksi - menginstall onnxruntime (CPU)...")
    !pip install -q onnxruntime

print("Dependencies installed!")

## 3. Jalankan Web Server

### Opsi A: Google Colab (dengan ngrok untuk akses publik)

Jalankan cell ini di Colab. Anda akan mendapat URL ngrok untuk mengakses web dari browser.

In [ ]:
# Install pyngrok (Colab/Kaggle)
!pip install -q pyngrok

import sys
import threading
import time
from pyngrok import ngrok

# Kaggle: Aktifkan "Add-ons" > "Internet" di Settings notebook
# Token ngrok (gratis di ngrok.com) - opsional untuk session lebih lama
ngrok.set_auth_token("1TfbVOS48SeXdQ7rJ2do5JjJFxG_4d5K3jMerctfbUsXvidrT")

# Redirect Flask log ke stdout agar tampil di Kaggle/Colab
import logging
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s - %(message)s')

def run_flask():
    import app
    app.app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

# Tunggu Flask siap
time.sleep(5)

# Buka tunnel ngrok
tunnel = ngrok.connect(5000)
url = tunnel.public_url if hasattr(tunnel, 'public_url') else str(tunnel)

# Tampilkan di output (penting untuk Kaggle)
print("=" * 60, flush=True)
print("Buka URL berikut di browser Anda:", flush=True)
print(url, flush=True)
print("=" * 60, flush=True)
sys.stdout.flush()

# Link klik untuk Kaggle/Colab
try:
    from IPython.display import display, HTML
    display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">🔗 Klik untuk membuka: {url}</a>'))
except Exception:
    pass

### Opsi B: Kaggle / Tanpa ngrok (akses lokal)

Jika di Kaggle atau tidak perlu akses dari luar, jalankan cell ini. Akses via URL yang ditampilkan di output (biasanya `http://127.0.0.1:5000`).

**Catatan Kaggle:** Kaggle Notebooks tidak mendukung ngrok dengan baik. Gunakan "Add-ons" > "Internet" untuk mengaktifkan akses internet, dan akses via "Output" > "Logs" untuk melihat URL.

In [ ]:
# Alternatif: Jalankan langsung tanpa ngrok
# Di Colab: Klik "Open in Colab" lalu "Preview" untuk melihat web di panel
# Di Kaggle: Output URL akan muncul di logs

import app
app.app.run(host='0.0.0.0', port=5000, debug=False)